In [0]:
"""Generating synthetic financial account time-series data."""

from __future__ import annotations

import numpy as np
import pandas as pd


RISK_STATE_NAMES = {
    0: "normal",
    1: "elevated_risk",
    2: "liquidity_stress",
    3: "delinquency_risk",
}


def _stress_to_state(stress: float) -> int:
    """Converting a latent financial-stress score into a risk-state class."""
    if stress < 0.35:
        return 0
    if stress < 0.55:
        return 1
    if stress < 0.75:
        return 2
    return 3


def generate_financial_timeseries(
    n_accounts: int = 2000,
    n_days: int = 60,
    start_date: str = "2026-01-01",
    random_seed: int = 42,
) -> pd.DataFrame:
    """
    Generating synthetic daily financial-account observations.

    Args:
        n_accounts:
            Number of independent customer accounts to simulate.
        n_days:
            Number of consecutive daily observations per account.
        start_date:
            First date in each account time series.
        random_seed:
            Seed used to make the simulation reproducible.

    Returns:
        A pandas DataFrame containing one row per account and day.
    """
    rng = np.random.default_rng(random_seed)

    dates = pd.date_range(
        start=start_date,
        periods=n_days,
        freq="D",
    )

    records: list[dict[str, object]] = []

    for account_number in range(1, n_accounts + 1):
        account_id = f"ACC_{account_number:05d}"

        account_type = rng.choice(
            ["checking", "credit", "business"],
            p=[0.50, 0.30, 0.20],
        )

        
        baseline_stress = float(
            rng.choice(
                [0.15, 0.40, 0.65, 0.85],
                p=[0.55, 0.25, 0.15, 0.05],
            )
        )

        stress = float(
            np.clip(
                rng.normal(
                    loc=baseline_stress,
                    scale=0.08,
                ),
                0.0,
                1.0,
            )
        )

        balance = float(
            rng.lognormal(
                mean=np.log(5000),
                sigma=0.45,
            )
        )

        credit_limit = float(rng.uniform(3000, 15000))

        for date in dates:
            # Evolving the latent financial-stress process.
            shock = rng.normal(
                loc=0.0,
                scale=0.05,
            )

            if rng.random() < 0.015:
                shock += rng.uniform(0.15, 0.35)
            
            stress = float(
                np.clip(
                    0.85 * stress
                    + 0.15 * baseline_stress
                    + shock,
                    0.0,
                    1.0,
                )
            )

            risk_state = _stress_to_state(stress)

            # Generating observed financial behavior conditional on stress.
            cash_inflow = max(
                0.0,
                rng.normal(
                    loc=180.0 * (1.0 - 0.45 * stress),
                    scale=60.0,
                ),
            )

            cash_outflow = max(
                0.0,
                rng.normal(
                    loc=150.0 * (1.0 + 0.55 * stress),
                    scale=55.0,
                ),
            )

            balance = max(
                0.0,
                balance + cash_inflow - cash_outflow,
            )

            transaction_count = max(
                0,
                int(
                    rng.poisson(
                        lam=max(
                            2.0,
                            10.0 - 3.0 * stress,
                        )
                    )
                ),
            )

            credit_utilization = float(
                np.clip(
                    rng.normal(
                        loc=0.25 + 0.65 * stress,
                        scale=0.08,
                    ),
                    0.0,
                    1.0,
                )
            )

            amount_due = max(
                20.0,
                credit_limit * credit_utilization * rng.uniform(0.015, 0.040),
            )

            expected_payment_ratio = np.clip(
                1.05 - 0.90 * stress,
                0.05,
                1.0,
            )

            payment_ratio = float(
                np.clip(
                    rng.normal(
                        loc=expected_payment_ratio,
                        scale=0.08,
                    ),
                    0.0,
                    1.0,
                )
            )

            amount_paid = amount_due * payment_ratio

            if stress < 0.45:
                days_past_due = 0
            else:
                days_past_due = max(
                    0,
                    int(
                        rng.normal(
                            loc=45 * stress,
                            scale=8,
                        )
                    ),
                )

            records.append(
                {
                    "account_id": account_id,
                    "date": date,
                    "account_type": account_type,
                    "balance": round(balance, 2),
                    "cash_inflow": round(cash_inflow, 2),
                    "cash_outflow": round(cash_outflow, 2),
                    "transaction_count": transaction_count,
                    "credit_utilization": round(
                        credit_utilization,
                        4,
                    ),
                    "amount_due": round(amount_due, 2),
                    "amount_paid": round(amount_paid, 2),
                    "payment_ratio": round(payment_ratio, 4),
                    "days_past_due": days_past_due,
                    "risk_state": risk_state,
                    "risk_state_name": RISK_STATE_NAMES[risk_state],
                }
            )

    return pd.DataFrame(records)


In [0]:
"""Injecting controlled data-quality issues into synthetic financial data."""


def inject_data_quality_issues(
    df: pd.DataFrame,
    missing_rate: float = 0.01,
    duplicate_rate: float = 0.005,
    invalid_rate: float = 0.002,
    random_seed: int = 123,
) -> pd.DataFrame:
    """
    Injecting controlled imperfections into clean synthetic financial data.

    Args:
        df:
            Clean synthetic financial time-series data.
        missing_rate:
            Fraction of rows selected for missing-value corruption
            in each chosen column.
        duplicate_rate:
            Fraction of rows duplicated.
        invalid_rate:
            Fraction of rows selected for deliberately invalid values.
        random_seed:
            Seed used to make corruption reproducible.

    Returns:
        A corrupted copy representing raw Bronze-style data.
    """
    rng = np.random.default_rng(random_seed)

    bronze_df = df.copy()

    n_rows = len(bronze_df)

    # Injecting missing values into selected numeric columns.
    numeric_missing_cols = [
        "balance",
        "cash_inflow",
        "cash_outflow",
        "credit_utilization",
        "payment_ratio",
    ]

    for column in numeric_missing_cols:
        n_missing = max(
            1,
            int(n_rows * missing_rate),
        )

        missing_indices = rng.choice(
            bronze_df.index,
            size=n_missing,
            replace=False,
        )

        bronze_df.loc[
            missing_indices,
            column,
        ] = np.nan

    # Injecting missing categorical values.
    n_missing_categories = max(
        1,
        int(n_rows * missing_rate),
    )

    category_indices = rng.choice(
        bronze_df.index,
        size=n_missing_categories,
        replace=False,
    )

    bronze_df.loc[
        category_indices,
        "account_type",
    ] = np.nan

    # Injecting impossible negative monetary values.
    n_invalid = max(
        1,
        int(n_rows * invalid_rate),
    )

    negative_indices = rng.choice(
        bronze_df.index,
        size=n_invalid,
        replace=False,
    )

    bronze_df.loc[
        negative_indices,
        "cash_outflow",
    ] = -np.abs(
        bronze_df.loc[
            negative_indices,
            "cash_outflow",
        ]
    )

    # Injecting invalid credit utilization above 100%.
    utilization_indices = rng.choice(
        bronze_df.index,
        size=n_invalid,
        replace=False,
    )

    bronze_df.loc[
        utilization_indices,
        "credit_utilization",
    ] = rng.uniform(
        1.05,
        1.50,
        size=n_invalid,
    )

    # Creating duplicate raw records.
    n_duplicates = max(
        1,
        int(n_rows * duplicate_rate),
    )

    duplicate_indices = rng.choice(
        bronze_df.index,
        size=n_duplicates,
        replace=False,
    )

    duplicate_rows = bronze_df.loc[duplicate_indices].copy()

    bronze_df = pd.concat(
        [
            bronze_df,
            duplicate_rows,
        ],
        ignore_index=True,
    )

    # Shuffling rows to mimic unordered ingestion.
    bronze_df = bronze_df.sample(
        frac=1.0,
        random_state=random_seed,
    ).reset_index(drop=True)

    return bronze_df


In [0]:
clean_df = generate_financial_timeseries()

bronze_pd = inject_data_quality_issues(clean_df)

print(clean_df.shape)
print(bronze_pd.shape)

In [0]:
# Converting pandas df (single-machine Python memory) into a spark df (Spark execution engine: potentially distributed/parallel computation)
bronze_spark = spark.createDataFrame(bronze_pd)

In [0]:
type(bronze_pd)

In [0]:
type(bronze_spark)

In [0]:
# A spark table's schema simply means its structure (not the Schema where the table lives!)
bronze_spark.printSchema()

In [0]:
display(bronze_spark.limit(20))

In [0]:
bronze_spark.show(20)

In [0]:
# Data ingestion: writing the `in-memory` spark dataframe (in temporary memory -- the spark session) into a `persisted` databricks table (in permanent storage -- the unity catalog) 
(
    bronze_spark.write
    .mode("overwrite")
    .saveAsTable(
        "finance_ml.bronze.account_events_raw"
    )
)

In [0]:

# Direct querying: query with spark.sql(...) in python when on databricks
spark.sql(
    """
    SHOW TABLES IN finance_ml.bronze
    """
).show(truncate=False)

In [0]:
result = spark.sql(
    """
    SELECT COUNT(*) AS row_count
    FROM finance_ml.bronze.account_events_raw
    """
)

display(result)

In [0]:
# DESCRIBE TABLE  →  columns/schema
display(
    spark.sql(
        """
        DESCRIBE TABLE finance_ml.bronze.account_events_raw
        """
    )
)

In [0]:
# DESCRIBE DETAIL    →   storage/table metadata
display(
    spark.sql(
        """
        DESCRIBE DETAIL finance_ml.bronze.account_events_raw
        """
    )
)

In [0]:
risk_distribution = spark.sql(
    """
    SELECT
        risk_state,
        risk_state_name,
        COUNT(*) AS n_rows
    FROM finance_ml.bronze.account_events_raw
    GROUP BY
        risk_state,
        risk_state_name
    ORDER BY risk_state
    """
)

display(risk_distribution)

In [0]:
duplicates = spark.sql(
    """
    SELECT
        account_id,
        date,
        COUNT(*) AS n_records
    FROM finance_ml.bronze.account_events_raw
    GROUP BY
        account_id,
        date
    HAVING COUNT(*) > 1
    """
)

display(duplicates)

In [0]:
duplicates.count()

In [0]:
invalid_values = spark.sql(
    """
    SELECT
        SUM(
            CASE
                WHEN cash_outflow < 0 THEN 1
                ELSE 0
            END
        ) AS negative_cash_outflows,

        SUM(
            CASE
                WHEN credit_utilization > 1 THEN 1
                ELSE 0
            END
        ) AS invalid_credit_utilization

    FROM finance_ml.bronze.account_events_raw
    """
)

display(invalid_values)

In [0]:
missing_values = spark.sql(
    """
    SELECT
        SUM(CASE WHEN account_type IS NULL THEN 1 ELSE 0 END)
            AS missing_account_type,

        SUM(CASE WHEN balance IS NULL THEN 1 ELSE 0 END)
            AS missing_balance,

        SUM(CASE WHEN cash_inflow IS NULL THEN 1 ELSE 0 END)
            AS missing_cash_inflow,

        SUM(CASE WHEN cash_outflow IS NULL THEN 1 ELSE 0 END)
            AS missing_cash_outflow,

        SUM(CASE WHEN credit_utilization IS NULL THEN 1 ELSE 0 END)
            AS missing_credit_utilization,

        SUM(CASE WHEN payment_ratio IS NULL THEN 1 ELSE 0 END)
            AS missing_payment_ratio

    FROM finance_ml.bronze.account_events_raw
    """
)

display(missing_values)